In [1]:
import fitz  # PyMuPDF
import cv2
import numpy as np
import os
import tensorflow as tf
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing import image

def extract_annotations(doc, color='red'):
    annotations_info = []
    color_map = {'red': ([0, 0, 250], [10, 10, 255])}
    for page_number in range(len(doc)):
        page = doc[page_number]
        pix = page.get_pixmap()
        if pix.samples:
            img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, 3)
            img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
            lower_color, upper_color = color_map[color] if color in color_map else ([0, 0, 0], [255, 255, 255])
            mask = cv2.inRange(img, np.array(lower_color), np.array(upper_color))
            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            for rect in [cv2.boundingRect(cnt) for cnt in contours]:
                x0, y0, w, h = rect
                annotations_info.append((page_number + 1, x0, y0, x0 + w, y0 + h))
    return annotations_info

def remove_border(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 240, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        contour = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(contour)
        img = img[y:y+h, x:x+w]
    return img, (x, y, w, h)

def extract_images(doc, annotations_info, output_folder, remove_border_flag=False):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    output_images, new_annotations_info = [], []
    for idx, (page_number, x0, y0, x1, y1) in enumerate(annotations_info):
        page = doc.load_page(page_number - 1)
        clip = fitz.Rect(x0, y0, x1, y1)
        pix = page.get_pixmap(clip=clip)
        img = cv2.imdecode(np.frombuffer(pix.tobytes(), dtype=np.uint8), cv2.IMREAD_COLOR)
        if remove_border_flag:
            img, (bx, by, bw, bh) = remove_border(img)
            new_annotations_info.append((page_number, x0 + bx, y0 + by, x0 + bx + bw, y0 + by + bh))
        else:
            new_annotations_info.append((page_number, x0, y0, x1, y1))
        output_images.append(img)
        cv2.imwrite(os.path.join(output_folder, f"extracted_image_{idx + 1}.png"), img)
    return output_images, new_annotations_info

def adjust_annotations_for_pdf2(original_annotations, source_rect, target_rect):
    adjusted_annotations = []
    x_scale = target_rect.width / source_rect.width
    y_scale = target_rect.height / source_rect.height
    for (page_number, x0, y0, x1, y1) in original_annotations:
        adjusted_annotations.append((page_number, x0 * x_scale, y0 * y_scale, x1 * x_scale, y1 * y_scale))
    return adjusted_annotations

def compute_vgg16_similarity(img1, img2):
    model = VGG16(weights='imagenet', include_top=False)
    model = Model(inputs=model.inputs, outputs=model.layers[-1].output)

    def get_features(img):
        img = cv2.resize(img, (224, 224))
        img = image.img_to_array(img)
        img = np.expand_dims(img, axis=0)
        img = preprocess_input(img)
        return model.predict(img).flatten()

    f1, f2 = get_features(img1), get_features(img2)
    return np.dot(f1, f2) / (np.linalg.norm(f1) * np.linalg.norm(f2))

def concatenate_images(img1, img2, similarity, output_path):
    height = min(img1.shape[0], img2.shape[0])
    img1_resized = cv2.resize(img1, (int(img1.shape[1] * height / img1.shape[0]), height))
    img2_resized = cv2.resize(img2, (int(img2.shape[1] * height / img2.shape[0]), height))
    concatenated_img = np.concatenate((img1_resized, img2_resized), axis=1)
    height, width = concatenated_img.shape[:2]
    concatenated_img = cv2.putText(concatenated_img, f'{similarity:.2f}', (width // 4, height - 10), 
                                   cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2, cv2.LINE_AA)
    cv2.imwrite(output_path, concatenated_img)

input_path1 = '/home/emanmunir/detection/vgg/2pages.pdf'
input_path2 = "/home/emanmunir/detection/vgg/Scan Colour .pdf"
doc1, doc2 = fitz.open(input_path1), fitz.open(input_path2)

annotations_info1 = extract_annotations(doc1)
output_folder1, output_folder2 = "/home/emanmunir/detection/vgg/vgg1", "/home/emanmunir/detection/vgg/vgg2"
output_images1, new_annotations_info1 = extract_images(doc1, annotations_info1, output_folder1, remove_border_flag=True)

source_rect, target_rect = doc1[0].rect, doc2[0].rect
adjusted_annotations_info2 = adjust_annotations_for_pdf2(annotations_info1, source_rect, target_rect)
output_images2, _ = extract_images(doc2, adjusted_annotations_info2, output_folder2, remove_border_flag=True)

concatenated_folder = "/home/emanmunir/detection/vgg/output"
if not os.path.exists(concatenated_folder):
    os.makedirs(concatenated_folder)

similarities = [compute_vgg16_similarity(img1, img2) for img1, img2 in zip(output_images1, output_images2)]
for idx, (img1, img2, similarity) in enumerate(zip(output_images1, output_images2, similarities)):
    output_path = os.path.join(concatenated_folder, f"concatenated_image_{idx + 1}.png")
    concatenate_images(img1, img2, similarity, output_path)

page_similarity_info = [(info[0], sim) for info, sim in zip(annotations_info1, similarities)]
print(f"Page numbers and similarities: {page_similarity_info}")


2024-07-11 17:59:39.592229: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-07-11 17:59:39.610739: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-07-11 17:59:39.634447: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-07-11 17:59:39.634488: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-07-11 17:59:39.652195: I tensorflow/core/platform/cpu_feature_gua

UnboundLocalError: local variable 'x' referenced before assignment

In [2]:
import fitz  # PyMuPDF
import cv2
import numpy as np
import os
import tensorflow as tf
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing import image

def extract_annotations(doc, color='red'):
    annotations_info = []
    color_map = {'red': ([0, 0, 250], [10, 10, 255])}
    for page_number in range(len(doc)):
        page = doc[page_number]
        pix = page.get_pixmap()
        if pix.samples:
            img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, 3)
            img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
            lower_color, upper_color = color_map[color] if color in color_map else ([0, 0, 0], [255, 255, 255])
            mask = cv2.inRange(img, np.array(lower_color), np.array(upper_color))
            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            for rect in [cv2.boundingRect(cnt) for cnt in contours]:
                x0, y0, w, h = rect
                annotations_info.append((page_number + 1, x0, y0, x0 + w, y0 + h))
    return annotations_info

def remove_border(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 240, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        contour = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(contour)
        img = img[y:y+h, x:x+w]
        return img, (x, y, w, h)
    return img, (0, 0, img.shape[1], img.shape[0])  # Default to no cropping if no contours

def extract_images(doc, annotations_info, output_folder, remove_border_flag=False):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    output_images, new_annotations_info = [], []
    for idx, (page_number, x0, y0, x1, y1) in enumerate(annotations_info):
        page = doc.load_page(page_number - 1)
        clip = fitz.Rect(x0, y0, x1, y1)
        pix = page.get_pixmap(clip=clip)
        img = cv2.imdecode(np.frombuffer(pix.tobytes(), dtype=np.uint8), cv2.IMREAD_COLOR)
        if remove_border_flag:
            img, (bx, by, bw, bh) = remove_border(img)
            new_annotations_info.append((page_number, x0 + bx, y0 + by, x0 + bx + bw, y0 + by + bh))
        else:
            new_annotations_info.append((page_number, x0, y0, x1, y1))
        output_images.append(img)
        cv2.imwrite(os.path.join(output_folder, f"extracted_image_{idx + 1}.png"), img)
    return output_images, new_annotations_info

def adjust_annotations_for_pdf2(original_annotations, source_rect, target_rect):
    adjusted_annotations = []
    x_scale = target_rect.width / source_rect.width
    y_scale = target_rect.height / source_rect.height
    for (page_number, x0, y0, x1, y1) in original_annotations:
        adjusted_annotations.append((page_number, x0 * x_scale, y0 * y_scale, x1 * x_scale, y1 * y_scale))
    return adjusted_annotations

def compute_vgg16_similarity(img1, img2):
    model = VGG16(weights='imagenet', include_top=False)
    model = Model(inputs=model.inputs, outputs=model.layers[-1].output)

    def get_features(img):
        img = cv2.resize(img, (224, 224))
        img = image.img_to_array(img)
        img = np.expand_dims(img, axis=0)
        img = preprocess_input(img)
        return model.predict(img).flatten()

    f1, f2 = get_features(img1), get_features(img2)
    return np.dot(f1, f2) / (np.linalg.norm(f1) * np.linalg.norm(f2))

def concatenate_images(img1, img2, similarity, output_path):
    height = min(img1.shape[0], img2.shape[0])
    img1_resized = cv2.resize(img1, (int(img1.shape[1] * height / img1.shape[0]), height))
    img2_resized = cv2.resize(img2, (int(img2.shape[1] * height / img2.shape[0]), height))
    concatenated_img = np.concatenate((img1_resized, img2_resized), axis=1)
    height, width = concatenated_img.shape[:2]
    concatenated_img = cv2.putText(concatenated_img, f'{similarity:.2f}', (width // 4, height - 10), 
                                   cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2, cv2.LINE_AA)
    cv2.imwrite(output_path, concatenated_img)

input_path1 = '/home/emanmunir/detection/vgg/2pages.pdf'
input_path2 = "/home/emanmunir/detection/vgg/Scan Colour .pdf"
doc1, doc2 = fitz.open(input_path1), fitz.open(input_path2)

annotations_info1 = extract_annotations(doc1)
output_folder1, output_folder2 = "/home/emanmunir/detection/vgg/vgg1", "/home/emanmunir/detection/vgg/vgg2"
output_images1, new_annotations_info1 = extract_images(doc1, annotations_info1, output_folder1, remove_border_flag=True)

source_rect, target_rect = doc1[0].rect, doc2[0].rect
adjusted_annotations_info2 = adjust_annotations_for_pdf2(annotations_info1, source_rect, target_rect)
output_images2, _ = extract_images(doc2, adjusted_annotations_info2, output_folder2, remove_border_flag=True)

concatenated_folder = "/home/emanmunir/detection/vgg/output"
if not os.path.exists(concatenated_folder):
    os.makedirs(concatenated_folder)

similarities = [compute_vgg16_similarity(img1, img2) for img1, img2 in zip(output_images1, output_images2)]
for idx, (img1, img2, similarity) in enumerate(zip(output_images1, output_images2, similarities)):
    output_path = os.path.join(concatenated_folder, f"concatenated_image_{idx + 1}.png")
    concatenate_images(img1, img2, similarity, output_path)

page_similarity_info = [(info[0], sim) for info, sim in zip(annotations_info1, similarities)]
print(f"Page numbers and similarities: {page_similarity_info}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 298ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 181ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 184ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 176ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 196ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step
Page numbers and similarities: [(1, 0.15218167), (1, 0.32974875), (2, 0.07500076), (2, 0.08367961), (2, 0.064863324)]


In [5]:
import os
import shutil
import cv2
import numpy as np
from pdf2image import convert_from_path
from scipy.ndimage import rotate
from PIL import Image
import pytesseract
import fitz  # PyMuPDF
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing import image

def correct_skew(image, delta=1, limit=12):
    def determine_score(arr, angle):
        data = rotate(arr, angle, reshape=False, order=0)
        histogram = np.sum(data, axis=1, dtype=float)
        score = np.sum((histogram[1:] - histogram[:-1]) ** 2, dtype=float)
        return histogram, score

    gray = cv2.cvtColor(np.array(image), cv2.COLOR_BGR2GRAY)
    thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 41, 15)
    scores = []
    angles = np.arange(-limit, limit + delta, delta)
    for angle in angles:
        histogram, score = determine_score(thresh, angle)
        scores.append(score)
    best_angle = angles[scores.index(max(scores))]
    return best_angle

def deskew_image(image, angle):
    (h, w) = image.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated = cv2.warpAffine(image, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
    return rotated

def detect_orientation(image):
    try:
        rgb_image = cv2.cvtColor(np.array(image), cv2.COLOR_BGR2RGB)
        osd = pytesseract.image_to_osd(rgb_image)
        rotate_angle = 0
        if "Rotate: 180" in osd:
            rotate_angle = 180
        return rotate_angle
    except Exception as e:
        return 0

def remove_white_margins(image, threshold=240):
    img = np.array(image)
    
    # If the image has color channels, convert it to grayscale by averaging the channels
    if img.ndim == 3:
        img = np.mean(img, axis=2)

    # Create a mask of pixels that are below the threshold (not white)
    mask = img < threshold

    # Find coordinates where the mask is true
    coords = np.argwhere(mask)

    # If no coordinates are found, return the original image
    if coords.size == 0:
        return image

    # Find the bounding box of the non-white pixels
    x0, y0 = coords.min(axis=0)
    x1, y1 = coords.max(axis=0) + 1

    # Crop the image to the bounding box
    return image.crop((y0, x0, y1, x1))


def process_pdf_file(file_path):
    images = convert_from_path(file_path)
    processed_images = []

    for image in images:
        rotate_angle = detect_orientation(image)
        if rotate_angle == 180:
            image = image.rotate(180, expand=True)
        angle = correct_skew(image)
        deskewed_image = deskew_image(np.array(image), angle)
        final_image = Image.fromarray(deskewed_image)
        #margin_removed_image = remove_white_margins(final_image)
        processed_images.append(final_image.convert('RGB'))

    output_path = os.path.splitext(file_path)[0] + "-processed.pdf"
    processed_images[0].save(output_path, save_all=True, append_images=processed_images[1:])
    return output_path

def extract_annotations(doc, color='red'):
    annotations_info = []
    color_map = {'red': ([0, 0, 250], [10, 10, 255])}

    for page_number in range(len(doc)):
        page = doc[page_number]
        pix = page.get_pixmap()

        if pix.samples:
            image = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, 3)
            image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

            lower_color, upper_color = color_map[color] if color in color_map else ([0, 0, 0], [255, 255, 255])
            lower_red = np.array(lower_color)
            upper_red = np.array(upper_color)

            mask = cv2.inRange(image, lower_red, upper_red)
            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

            for rect in [cv2.boundingRect(cnt) for cnt in contours]:
                x0, y0, w, h = rect
                x1, y1 = x0 + w, y0 + h
                annotations_info.append((page_number + 1, x0, y0, x1, y1))

    return annotations_info

def remove_border(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 240, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        contour = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(contour)
        img = img[y:y+h, x:x+w]
    return img, (x, y, w, h)

def extract_images(doc, annotations_info, output_folder, remove_border_flag=False):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    output_images = []
    new_annotations_info = []
    for idx, (page_number, x0, y0, x1, y1) in enumerate(annotations_info):
        page = doc.load_page(page_number - 1)
        clip = fitz.Rect(x0, y0, x1, y1)
        pix = page.get_pixmap(clip=clip)
        img = cv2.imdecode(np.frombuffer(pix.tobytes(), dtype=np.uint8), cv2.IMREAD_COLOR)
        if remove_border_flag:
            img, (border_x, border_y, border_w, border_h) = remove_border(img)
            new_x0 = x0 + border_x
            new_y0 = y0 + border_y
            new_x1 = x0 + border_x + border_w
            new_y1 = y0 + border_y + border_h
            new_annotations_info.append((page_number, new_x0, new_y0, new_x1, new_y1))
        else:
            new_annotations_info.append((page_number, x0, y0, x1, y1))
        output_images.append(img)
        img_path = os.path.join(output_folder, f"extracted_image_{idx + 1}.png")
        cv2.imwrite(img_path, img)
    return output_images, new_annotations_info

def adjust_annotations_for_pdf2(original_annotations, source_rect, target_rect):
    adjusted_annotations = []
    x_scale = target_rect.width / source_rect.width
    y_scale = target_rect.height / source_rect.height
    for (page_number, x0, y0, x1, y1) in original_annotations:
        new_x0 = x0 * x_scale
        new_y0 = y0 * y_scale
        new_x1 = x1 * x_scale
        new_y1 = y1 * y_scale
        adjusted_annotations.append((page_number, new_x0, new_y0, new_x1, new_y1))
    return adjusted_annotations

def compute_vgg16_similarity(img1, img2):
    model = VGG16(weights='imagenet', include_top=False)
    model = Model(inputs=model.inputs, outputs=model.layers[-1].output)

    def get_features(img):
        img = cv2.resize(img, (224, 224))
        img = image.img_to_array(img)
        img = np.expand_dims(img, axis=0)
        img = preprocess_input(img)
        return model.predict(img).flatten()

    f1, f2 = get_features(img1), get_features(img2)
    return np.dot(f1, f2) / (np.linalg.norm(f1) * np.linalg.norm(f2))

# Usage example:
template_file_path = '/home/emanmunir/detection/vgg/2pages.pdf'
scanned_file_path = '/home/emanmunir/detection/vgg/Scan BW.pdf'
threshold = 0.8

# Process scanned PDF
processed_scanned_path = process_pdf_file(scanned_file_path)

# Open processed and template PDFs
doc_template = fitz.open(template_file_path)
doc_scanned = fitz.open(processed_scanned_path)

# Extract annotations and images
annotations_info_template = extract_annotations(doc_template)
output_folder_template = "/home/emanmunir/detection/vgg/vgg1"
output_folder_scanned = "/home/emanmunir/detection/vgg/vgg2"
output_images_template, new_annotations_info_template = extract_images(doc_template, annotations_info_template, output_folder_template, remove_border_flag=True)

source_rect = doc_template[0].rect
target_rect = doc_scanned[0].rect
adjusted_annotations_scanned = adjust_annotations_for_pdf2(annotations_info_template, source_rect, target_rect)
output_images_scanned, _ = extract_images(doc_scanned, adjusted_annotations_scanned, output_folder_scanned, remove_border_flag=True)

# Compare images and print results
page_results = {}
for idx, (img_template, img_scanned, annotation) in enumerate(zip(output_images_template, output_images_scanned, annotations_info_template)):
    score = compute_vgg16_similarity(img_template, img_scanned)
    is_signed = score < threshold
    page_number = annotation[0]
    if page_number not in page_results:
        page_results[page_number] = {"signed": 0, "unsigned": 0}
    if is_signed:
        page_results[page_number]["signed"] += 1
    else:
        page_results[page_number]["unsigned"] += 1

result = [
    f"page no. {page} has {info['signed']} signed boxes and {info['unsigned']} unsigned boxes"
    for page, info in page_results.items()
]

print(result)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 195ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 211ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 203ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 131ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 204ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 148ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 266ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step
['page no. 1 has 2 signed boxes and 0 unsigned boxes', 'page no. 2 has 3 signed boxes and 0 unsigned boxes']
